# CFEVER：三階段事實查核作業

本範例的三個階段都使用中文 BERT：

1. **Document Retrieval**：找出與 claim 相關的 Wikipedia 頁面。
2. **Evidence Retrieval**：從這些頁面挑選證據句子。
3. **Classification**：讓 BERT 根據 claim 和證據預測標籤。

所有程式碼已完整提供，請由上而下執行。
為了縮短課堂測試時間，語料只使用公開訓練集涉及的頁面與證據句。
驗證集用於觀察表現，測試集只用來產生提交結果。



## 1. 套件與參數

請先啟用 GPU。優先從 Kaggle Input 載入完整模型，便可離線執行。
第一次從 Hugging Face 下載時需要 Internet；下載後須把模型加入 Input，供正式重跑使用。



In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# 固定亂數種子，讓抽樣與模型初始化較容易重現。
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置：", device)



In [ ]:
# Kaggle 從 Input 尋找資料；本機預設讀取目前資料夾。
# 若有多份同名資料，可把 DATA_DIR 改成這次競賽的資料夾。
if Path("/kaggle/input").exists():
    DATA_DIR = Path("/kaggle/input")
    OUTPUT_DIR = Path("/kaggle/working")
    MODEL_DIR = None  # 自動尋找 Input 模型；也可填入模型資料夾的完整路徑。
else:
    DATA_DIR = Path(".")
    OUTPUT_DIR = Path("bert_student_output")
    MODEL_DIR = Path("bert-base-chinese")

# 可處理 Kaggle Input 中多層的 Dataset 目錄。
def find_data_file(filename):
    folder = Path(DATA_DIR)
    direct_path = folder / filename
    if direct_path.is_file():
        return direct_path

    matches = sorted(path for path in folder.rglob(filename) if path.is_file())
    if not matches:
        raise FileNotFoundError(
            f"找不到 {filename}。請加入競賽資料，或修改 DATA_DIR。目前搜尋目錄：{folder}"
        )
    if len(matches) > 1:
        paths = "\n".join(str(path) for path in matches)
        raise ValueError(f"找到多份 {filename}，請把 DATA_DIR 改成指定的資料夾：\n{paths}")
    return matches[0]


TRAIN_PATH = find_data_file("train.jsonl")
DEV_PATH = find_data_file("dev.jsonl")
TEST_PATH = find_data_file("test.jsonl")
WIKI_PATH = find_data_file("wiki_pages.jsonl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("訓練資料：", TRAIN_PATH)
print("驗證資料：", DEV_PATH)
print("測試資料：", TEST_PATH)
print("Wikipedia：", WIKI_PATH)

TRAIN_SAMPLES = 3000    # 設為 None，表示使用全部訓練資料。
DEV_SAMPLES = 300      # 設為 None，表示使用全部驗證資料。
DOCUMENT_TOP_K = 5     # 第一階段預設保留前 5 個頁面。
EMBEDDING_BATCH_SIZE = 16  # 每批編碼幾段文字；GPU 記憶體不足時可調小。
RETRIEVAL_MAX_LENGTH = 256 # 頁面與 claim 編碼的 token 上限，不能超過 BERT 的 512。
EVIDENCE_TOP_K = 5     # 第二階段保留幾個句子；目前 metric 最多計算前 5 句。
EPOCHS = 1
BATCH_SIZE = 8
MAX_LENGTH = 192       # claim 與證據合計的 token 上限。
LEARNING_RATE = 2e-5
ALLOW_MODEL_DOWNLOAD = True  # 首次下載需開啟 Internet；離線提交時可設為 False。



### 讀取資料

JSONL 每一行是一筆 JSON。`label` 是標準標籤，`evidence` 是標準證據群組。
`all_train_df` 保留全部公開訓練資料，供建立檢索語料使用。
`train_df` 是本次用來訓練 BERT 的抽樣資料。
使用開啟的檔案物件讀取，避免把路徑字串誤判為 JSON 內容。



In [ ]:
with Path(TRAIN_PATH).open(encoding="utf-8-sig") as file:
    all_train_df = pd.read_json(file, lines=True)
train_df = all_train_df.copy()

with Path(DEV_PATH).open(encoding="utf-8-sig") as file:
    dev_df = pd.read_json(file, lines=True)

# 測試時只保留輸入欄位；模型不使用測試集標準答案。
with Path(TEST_PATH).open(encoding="utf-8-sig") as file:
    test_df = pd.read_json(file, lines=True)[["id", "claim"]]

if TRAIN_SAMPLES is not None:
    train_df = train_df.sample(n=min(TRAIN_SAMPLES, len(train_df)), random_state=SEED)
if DEV_SAMPLES is not None:
    dev_df = dev_df.sample(n=min(DEV_SAMPLES, len(dev_df)), random_state=SEED)

# 重設索引後，第 i 筆資料就能對應到第 i 個檢索結果。
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("訓練／驗證／測試筆數：", len(train_df), len(dev_df), len(test_df))
train_df[["id", "claim", "label"]].head()



### 下載中文 BERT

先尋找 Input 中的完整 BERT 模型，找不到才考慮下載至工作目錄。
`/kaggle/working` 的互動執行檔案不會自動變成下一次提交的 Input。
`.part` 是尚未完成的下載檔；完整下載後才改成正式檔名。
如需離線重跑，請將模型資料夾以 Dataset 或 Notebook Output 加入 Input。



In [ ]:
MODEL_NAME = "google-bert/bert-base-chinese"
MODEL_VERSION = "8f23c25b06e129b6c986331a13d8d025a92cf0ea"
model_files = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]

if MODEL_DIR is None:
    model_folders = []
    for config_path in Path("/kaggle/input").rglob("config.json"):
        folder = config_path.parent
        if all((folder / filename).is_file() for filename in model_files):
            config = json.loads(config_path.read_text(encoding="utf-8"))
            if config.get("model_type") == "bert":
                model_folders.append(folder)
    if len(model_folders) > 1:
        raise ValueError(f"Input 中有多個 BERT 模型，請設定 MODEL_DIR：{model_folders}")
    MODEL_DIR = model_folders[0] if model_folders else OUTPUT_DIR / "bert-base-chinese"

MODEL_DIR = Path(MODEL_DIR)
missing_files = [name for name in model_files if not (MODEL_DIR / name).is_file()]
if missing_files:
    # Input 是唯讀目錄；模型不完整時，請重新附加完整模型，不能在該目錄下載。
    if MODEL_DIR.resolve().is_relative_to(Path("/kaggle/input").resolve()):
        raise FileNotFoundError(f"Input 模型缺少檔案：{missing_files}。模型目錄：{MODEL_DIR}")
    if not ALLOW_MODEL_DOWNLOAD:
        raise FileNotFoundError("找不到完整 BERT 模型。請將模型以 Dataset 加入 Input，或開啟 Internet 允許下載。")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

for filename in missing_files:
    model_path = MODEL_DIR / filename
    if not model_path.exists():
        print("下載：", filename)
        url = f"https://huggingface.co/{MODEL_NAME}/resolve/{MODEL_VERSION}/{filename}"
        temporary_path = MODEL_DIR / (filename + ".part")
        try:
            urllib.request.urlretrieve(url, temporary_path)
        except OSError as error:
            raise RuntimeError(
                "無法下載 BERT。若提交環境禁止 Internet，請先將完整模型加入 Kaggle Input。"
            ) from error
        temporary_path.replace(model_path)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
print("模型目錄：", MODEL_DIR)



## 2. Document Retrieval：找相關頁面

每筆標準證據是 `[annotation_id, evidence_id, page_title, sentence_id]`。
先收集公開訓練資料涉及的頁面，再從 Wikipedia 讀取這些頁面的文字。
本範例只索引這些頁面，因此找不到的頁面會限制後續表現。



In [ ]:
wanted_pages = set()
wanted_sentences = set()

for row in all_train_df.itertuples():
    # NOT ENOUGH INFO 沒有可用的標準證據句。
    if row.label.upper() == "NOT ENOUGH INFO":
        continue
    for group in row.evidence:
        for item in group:
            page_title = item[2]
            sentence_id = item[3]
            wanted_pages.add(page_title)
            wanted_sentences.add((page_title, sentence_id))

# Wikipedia 檔案較大，逐行讀取，僅保留需要的頁面。
wiki_pages = {}
with WIKI_PATH.open(encoding="utf-8") as file:
    for line in file:
        page = json.loads(line)
        if page["id"] in wanted_pages:
            wiki_pages[page["id"]] = page

print("候選頁面數：", len(wiki_pages))



### 用 BERT 將文字轉成向量

BERT 為每個 token 產生一個向量，這裡用 **mean pooling** 將它們平均成文字向量。
平均時使用 attention mask 排除 padding，避免補齊長度的 token 影響結果。
再用 **L2 normalization** 把向量長度變成 1，這樣內積就等於 cosine similarity。

頁面與 claim 使用同一個預訓練 BERT，這個檢索模型不參與後面的分類器訓練。
每頁只編碼標題與開頭文字，超過 `RETRIEVAL_MAX_LENGTH` 的 token 會被截斷。
同一個 encode_texts 函式會用於頁面、claim 和證據句的編碼。
原始 BERT 未針對檢索做對比學習訓練，這裡用來示範向量檢索。



In [ ]:
retrieval_model = AutoModel.from_pretrained(str(MODEL_DIR), local_files_only=True)
retrieval_model = retrieval_model.to(device)
retrieval_model.eval()  # 關閉 dropout，讓相同文字的向量保持穩定。
retrieval_model.requires_grad_(False)  # 此階段只產生向量，不更新參數。

def encode_texts(texts):
    """將一批文字轉為長度為 1 的 BERT 向量，回傳 CPU tensor。"""
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), EMBEDDING_BATCH_SIZE):
            batch_texts = texts[start:start + EMBEDDING_BATCH_SIZE]
            inputs = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=RETRIEVAL_MAX_LENGTH, return_tensors="pt",
            )
            for name in inputs:
                inputs[name] = inputs[name].to(device)

            # token_vectors 的形狀是 [batch 大小, token 數, 768]。
            token_vectors = retrieval_model(**inputs).last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1).float()

            # 只平均 attention_mask = 1 的 token；padding 的權重是 0。
            vector_sum = (token_vectors * mask).sum(dim=1)
            token_count = mask.sum(dim=1).clamp(min=1)
            text_vectors = vector_sum / token_count
            text_vectors = F.normalize(text_vectors, p=2, dim=1)
            embeddings.append(text_vectors.cpu())

    return torch.cat(embeddings, dim=0)




### 建立頁面向量索引

每個頁面只需要編碼一次，之後所有 claim 都與這些固定向量比較。
此處的索引就是一個矩陣，每一列對應 `page_titles` 中的一個頁面。
BERT-base 的向量維度為 768，索引保留在 CPU，以節省 GPU 記憶體。



In [ ]:
page_titles = sorted(wiki_pages)
document_texts = []
for title in page_titles:
    text = title + " " + wiki_pages[title]["text"][:6000]
    document_texts.append(text)

document_embeddings = encode_texts(document_texts)
print("頁面向量矩陣大小：", document_embeddings.shape)



### 用 cosine similarity 找最相關的頁面

把 claim 也轉成單位向量，再計算 `claim_embeddings @ document_embeddings.T`。
結果矩陣的每一列是一個 claim，每一欄是一個頁面；數值越大，向量方向越接近。
`topk` 選出每個 claim 相似度最高的頁面。Cosine similarity 可為負數，仍依排名取前 K 個。
claim 向量只需計算一次，下一階段選句時會重用。



In [ ]:
def retrieve_documents(claim_embeddings, top_k):
    """每個 claim 回傳一份頁面名稱清單，順序為相似度由高到低。"""
    similarities = claim_embeddings @ document_embeddings.T
    count = min(top_k, len(page_titles))
    top_indices = similarities.topk(k=count, dim=1).indices.tolist()

    results = []
    for indices in top_indices:
        pages = []
        for index in indices:
            pages.append(page_titles[index])
        results.append(pages)
    return results

train_claim_embeddings = encode_texts(train_df["claim"].tolist())
dev_claim_embeddings = encode_texts(dev_df["claim"].tolist())
test_claim_embeddings = encode_texts(test_df["claim"].tolist())

train_documents = retrieve_documents(train_claim_embeddings, DOCUMENT_TOP_K)
dev_documents = retrieve_documents(dev_claim_embeddings, DOCUMENT_TOP_K)
test_documents = retrieve_documents(test_claim_embeddings, DOCUMENT_TOP_K)

print("測試 claim：", test_df.loc[0, "claim"])
print("取回頁面：", test_documents[0])



### Document Retrieval Recall@5

用有標準答案的驗證集計分，只統計 supports、refutes 題目。
前 5 個頁面必須包含任一完整證據組需要的所有頁面，該題才算命中。
若一組證據需要 A、B 兩頁，只找到 A 仍算未命中。此分數不考慮分類結果。



In [ ]:
document_hits = 0
document_count = 0
for index, row in enumerate(dev_df.itertuples()):
    if row.label.upper() == "NOT ENOUGH INFO":
        continue
    document_count += 1
    predicted_pages = set(dev_documents[index][:5])

    for group in row.evidence:
        gold_pages = set()
        for item in group:
            gold_pages.add(item[2])
        if gold_pages.issubset(predicted_pages):
            document_hits += 1
            break  # 任一組完整命中即可；同一題不重複加分。

document_recall_at5 = document_hits / max(document_count, 1)
print(f"Document Retrieval Recall@5: {document_recall_at5:.4f}")



## 3. Evidence Retrieval：從頁面挑選句子

第一階段回答「看哪些頁面」，第二階段回答「使用哪些句子」。
候選句子只使用公開訓練集標註過的句子，並保留 Wikipedia 的原始句子編號。
例如跳過空白的第 1 句後，第 2 句仍然是第 2 句，不可以重新編號。
同樣用 BERT、mean pooling 和 L2 normalization 建立句子向量，再算 cosine similarity。



In [ ]:
sentence_info = []     # 第 i 個元素是 (頁面名稱, 原始句子編號)。
sentence_texts = []    # 第 i 個元素是對應句子的文字。

for title in page_titles:
    for line in wiki_pages[title]["lines"].splitlines():
        parts = line.split("\t")
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        sentence_id = int(parts[0])
        text = parts[1].strip()
        if text and (title, sentence_id) in wanted_sentences:
            sentence_info.append((title, sentence_id))
            sentence_texts.append(text)

sentence_embeddings = encode_texts(sentence_texts)
print("候選證據句數：", len(sentence_info))
print("句子向量矩陣大小：", sentence_embeddings.shape)



In [ ]:
def retrieve_evidence(claim_embeddings, document_results, top_k):
    """回傳證據座標，以及給 BERT 閱讀的句子文字。"""
    all_evidence = []
    all_contexts = []

    for claim_vector, pages in zip(claim_embeddings, document_results):
        # 只允許從第一階段取回的頁面選句。
        candidate_indices = []
        for index, (title, sentence_id) in enumerate(sentence_info):
            if title in pages:
                candidate_indices.append(index)

        evidence = []
        selected_texts = []
        if candidate_indices:
            # claim 與候選句都已正規化，所以內積就是 cosine similarity。
            candidates = sentence_embeddings[candidate_indices]
            scores = candidates @ claim_vector
            count = min(top_k, len(candidate_indices))
            ranked_indices = scores.topk(k=count).indices.tolist()

            for local_index in ranked_indices:
                index = candidate_indices[local_index]
                title, sentence_id = sentence_info[index]
                evidence.append([title, sentence_id])
                selected_texts.append(sentence_texts[index])

        all_evidence.append(evidence)
        all_contexts.append(" ".join(selected_texts)[:1000])
    return all_evidence, all_contexts

train_evidence, train_contexts = retrieve_evidence(train_claim_embeddings, train_documents, EVIDENCE_TOP_K)
dev_evidence, dev_contexts = retrieve_evidence(dev_claim_embeddings, dev_documents, EVIDENCE_TOP_K)
test_evidence, test_contexts = retrieve_evidence(test_claim_embeddings, test_documents, EVIDENCE_TOP_K)

print("證據座標：", test_evidence[0])
print("證據文字：", test_contexts[0])



### Evidence Retrieval Recall@5

同樣只統計 supports、refutes，檢查前 5 個證據句是否完整涵蓋任一標準證據組。
每句必須同時比對頁面名稱與原始句子編號；部分命中不給分。
此分數評估實際流程（先找頁面，再選句），不假設第一階段已找到正確頁面。



In [ ]:
evidence_hits = 0
evidence_count = 0
for index, row in enumerate(dev_df.itertuples()):
    if row.label.upper() == "NOT ENOUGH INFO":
        continue
    evidence_count += 1
    predicted_sentences = set(map(tuple, dev_evidence[index][:5]))

    for group in row.evidence:
        gold_sentences = set()
        for item in group:
            gold_sentences.add((item[2], item[3]))
        if gold_sentences.issubset(predicted_sentences):
            evidence_hits += 1
            break

evidence_recall_at5 = evidence_hits / max(evidence_count, 1)
print(f"Evidence Retrieval Recall@5: {evidence_recall_at5:.4f}")



## 4. Classification：用 BERT 預測標籤

將 claim 與證據作為一組文字配對：`[CLS] claim [SEP] context [SEP]`。
tokenizer 把文字轉為 token ID、attention mask 等模型輸入。
模型的三個輸出分別對應 supports、refutes、NOT ENOUGH INFO。



In [ ]:
labels = ["supports", "refutes", "NOT ENOUGH INFO"]
label_to_id = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
train_labels = train_df["label"].str.upper().map(label_to_id).tolist()

class ClaimDataset(Dataset):
    def __init__(self, claims, contexts, target_labels=None):
        # 固定長度 padding，讓 DataLoader 可以直接把多筆資料堆成一個 batch。
        self.encodings = tokenizer(
            list(claims), contexts, truncation=True,
            padding="max_length", max_length=MAX_LENGTH,
        )
        self.labels = target_labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        item = {}
        for name, values in self.encodings.items():
            item[name] = torch.tensor(values[index], dtype=torch.long)
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

train_dataset = ClaimDataset(train_df["claim"], train_contexts, train_labels)
dev_dataset = ClaimDataset(dev_df["claim"], dev_contexts)
test_dataset = ClaimDataset(test_df["claim"], test_contexts)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



### 模型與 Optimizer

`AutoModelForSequenceClassification` 會在預訓練 BERT 上加入三分類層。
分類層是新初始化的，因此載入時看到相關提示是正常的，需要用下面的資料訓練它。
傳入 `labels` 時，模型會自動計算 Cross Entropy loss。



In [ ]:
# 比較不同檢索方法時，使用相同的分類器初始化。
torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR), num_labels=3, local_files_only=True,
    id2label=dict(enumerate(labels)),
    label2id={label: index for index, label in enumerate(labels)},
)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)



### Training

每批資料依序執行：清空梯度、計算預測與 loss、反向傳播、更新參數。
`model.train()` 會啟用訓練模式；gradient clipping 用來限制過大的梯度。



In [ ]:
training_losses = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader, start=1):
        # 將這個 batch 的每個 tensor 搬到 GPU（或 CPU）。
        for name in batch:
            batch[name] = batch[name].to(device)

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        if step % 50 == 0 or step == len(train_loader):
            print(f"Epoch {epoch + 1}, step {step}, average loss = {total_loss / step:.4f}")

    training_losses.append(total_loss / len(train_loader))

model.save_pretrained(OUTPUT_DIR / "finetuned_bert")
tokenizer.save_pretrained(OUTPUT_DIR / "finetuned_bert")



### Prediction

`model.eval()` 關閉訓練模式，`torch.no_grad()` 表示不計算梯度。
`argmax` 選出分數最大的類別編號，再轉回標籤文字。



In [ ]:
def predict_labels(data_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            for name in batch:
                batch[name] = batch[name].to(device)
            logits = model(**batch).logits
            predicted_ids = logits.argmax(dim=1).cpu().tolist()
            for label_id in predicted_ids:
                predictions.append(labels[label_id])
    return predictions

dev_predictions = predict_labels(dev_loader)
test_predictions = predict_labels(test_loader)
print("前 5 筆測試預測：", test_predictions[:5])



### Classification Accuracy

只比對分類標籤，不要求證據正確；所有題目（包含 NEI）都進入分母。
因此可以單獨觀察 BERT 分類器的表現，不會與檢索分數混在一起。



In [ ]:
label_hits = 0
for prediction, gold_label in zip(dev_predictions, dev_df["label"]):
    if prediction.upper() == gold_label.upper():
        label_hits += 1

classification_accuracy = label_hits / len(dev_df)
print(f"Classification Accuracy: {classification_accuracy:.4f}")



## 5. Validation：分開觀察三階段的表現

Document Recall、Evidence Recall 都要求完整涵蓋任一組標準證據，且只統計可驗證的題目。
Label Accuracy 只看分類；CFEVER Score 還要求證據正確，NEI 則只看標籤。
三個階段已各自印出分數；下面彙整，並額外計算競賽使用的 CFEVER Score。
公開測試集沒有標準答案，因此分數在 dev.jsonl 上計算，test.jsonl 只輸出預測。
所有分數的範圍都是 0～1，例如 0.8 表示 80%。



In [ ]:
strict_hits = 0

for index, row in enumerate(dev_df.itertuples()):
    correct_label = dev_predictions[index].upper() == row.label.upper()
    if row.label.upper() == "NOT ENOUGH INFO":
        strict_hits += correct_label
        continue

    # 與 Kaggle metric 一致：即使取回更多句，也只計算前 5 句。
    predicted_sentences = set(map(tuple, dev_evidence[index][:5]))
    for group in row.evidence:
        gold_sentences = set()
        for item in group:
            gold_sentences.add((item[2], item[3]))
        if correct_label and gold_sentences.issubset(predicted_sentences):
            strict_hits += 1
            break

metrics = {
    "document_recall_at5": document_recall_at5,
    "evidence_recall_at5": evidence_recall_at5,
    "classification_accuracy": classification_accuracy,
    "cfever_score": strict_hits / len(dev_df),
}
print(f"Document Retrieval Recall@5: {document_recall_at5:.4f}")
print(f"Evidence Retrieval Recall@5: {evidence_recall_at5:.4f}")
print(f"Classification Accuracy: {classification_accuracy:.4f}")
print(f"CFEVER Score: {metrics['cfever_score']:.4f}")



## 6. 產生 submission.csv

提交檔需要 `id`、`predicted_label`、`predicted_evidence` 三欄。
證據使用 `json.dumps()` 轉成合法 JSON 字串，NEI 的證據填 `[]`。
保持測試資料的 ID 與順序，再從 Kaggle Notebook 輸出提交這份 CSV。



In [ ]:
evidence_strings = []
for label, evidence in zip(test_predictions, test_evidence):
    if label == "NOT ENOUGH INFO":
        evidence = []
    evidence_strings.append(json.dumps(evidence, ensure_ascii=False))

submission = pd.DataFrame({
    "id": test_df["id"],
    "predicted_label": test_predictions,
    "predicted_evidence": evidence_strings,
})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False, encoding="utf-8")
print("輸出筆數：", len(submission))
print("輸出位置：", OUTPUT_DIR / "submission.csv")
submission.head()



# Report

Your Kaggle username: 

Your public leaderboard score:

Your private leaderboard score:


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer:

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer:

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer:

4. … Anything that can strengthen your report. (10%)

Answer:

5. Generative AI Usage Statement

Answer: